In [18]:
%cd practicas/

[Errno 2] No such file or directory: 'practicas/'
/workspace/practicas


/usr/local/lib/python3.10/site-packages/IPython/core/magics/osm.py:393: UserWarning: This is now an optional IPython functionality, using bookmarks requires you to install the `pickleshare` library.
  bkms = self.shell.db.get('bookmarks', {})


In [19]:
from pyspark.sql import SparkSession

spark = ( SparkSession.builder
         .appName("pr503")
         .master("spark://spark-master:7077")
         .getOrCreate()
         )
 
sc = spark.sparkContext

In [20]:
from pyspark.sql.types import StructType, StructField, BooleanType, IntegerType, StringType, DoubleType, LongType, TimestampType
from pyspark.sql import functions as f

schema_world = StructType([
    StructField("index", IntegerType(), True),
    StructField("Title", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Amount(in rupees)", StringType(), True),
    StructField("Price (in rupees)e", IntegerType(), True),
    StructField("location", StringType(), True),
    StructField("Carpet Area", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Floor", StringType(), True),
    StructField("Transaction", StringType(), True),
    StructField("Furnishing", StringType(), True),
    StructField("facing", StringType(), True),
    StructField("overlooking", StringType(), True),
    StructField("Society", StringType(), True),
    StructField("Bathroom", IntegerType(), True),
    StructField("Balcony", IntegerType(), True),
    StructField("Car Parking", StringType(), True),
    StructField("Ownership", StringType(), True),
    StructField("Super Area", StringType(), True),
    StructField("Dimensions", StringType(), True),
    StructField("Plot Area", StringType(), True),
    ])

df = (spark.read
             .format("csv")
             .schema(schema_world)
             .option("header", "True")
             .option("sep", ";")
             .load("./data/house_prices.csv"))
df.show(5)

+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+
|index|               Title|         Description|Amount(in rupees)|Price (in rupees)e|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|
+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+
|    0|1 BHK Ready to Oc...|Bhiwandi, Thane h...|          42 Lac |              6000|   thane|   500 sqft|Ready to Move|

## 1.1 Estandarizacion monetaria (de INR a USD)

In [21]:
split_col = f.split(f.col("Amount(in rupees)"), " ")

df = df.withColumn(
    "Amount(in rupees)",
    f.when(split_col.getItem(1) == "Lac", 
           split_col.getItem(0).cast("double") * 100000)
     .otherwise(
           split_col.getItem(0).cast("double") * 10000000
     )
)
df.show(5)

+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+
|index|               Title|         Description|Amount(in rupees)|Price (in rupees)e|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|
+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+
|    0|1 BHK Ready to Oc...|Bhiwandi, Thane h...|        4200000.0|              6000|   thane|   500 sqft|Ready to Move|

In [22]:
df = df.withColumn("Amount_USD", (f.col("Amount(in rupees)")) * 0.012)
df.show(5)

+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+
|index|               Title|         Description|Amount(in rupees)|Price (in rupees)e|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|
+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+
|    0|1 BHK Ready to Oc...|Bhiwandi, Thane h...|        4200000.0|              6000|  

## 1.2 Estandarización de superficie

In [23]:
df = df.withColumn("Carpet Area", f.split(f.col("Carpet Area"), " ").getItem(0).cast("float"))
df.show(5)

+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+
|index|               Title|         Description|Amount(in rupees)|Price (in rupees)e|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|
+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+
|    0|1 BHK Ready to Oc...|Bhiwandi, Thane h...|        4200000.0|              6000|  

In [24]:
df = df.withColumn("Area_m2", f.col("Carpet Area") * 0.0929)
df.show(5)

+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+
|index|               Title|         Description|Amount(in rupees)|Price (in rupees)e|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|
+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+
|    0|1 BHK Ready to Oc...|Bhi

# 2. Objetivos de análisis estadístico
## 2.1 Medidas de despersión (varianza y desviación estándar)

In [25]:
desviacion_amount = df.select(f.stddev("Amount_USD")).collect()[0][0]
varianza_amount = df.select(f.variance("Amount_USD")).collect()[0][0]

media_amount = df.select(f.mean("Amount_USD")).collect()[0][0]
mediana_amount = df.select(f.median("Amount_USD")).collect()[0][0]


desviacion_area = df.select(f.stddev("Area_m2")).collect()[0][0]
varianza_area = df.select(f.variance("Area_m2")).collect()[0][0]

media_area = df.select(f.mean("Area_m2")).collect()[0][0]
mediana_area = df.select(f.median("Area_m2")).collect()[0][0]


print(f"Desviacion: {desviacion_amount}")
print(f"Varianza: {varianza_amount}")

Desviacion: 473259.23319444305
Varianza: 223974301803.79224


- Si la desviación es muy alta quiere decir que los precios de mercado son muy dispares para confiar en el promedio

## 2.2 Medidas de Forma (Skewness y Kurtosis)

In [26]:
skewness = df.select(f.skewness("Amount_USD")).collect()[0]
kurtosis = df.select(f.kurtosis("Amount_USD")).collect()[0]

print(f"Desviacion: {skewness}")
print(f"Varianza: {kurtosis}")

Desviacion: Row(skewness(Amount_USD)=270.25366063569174)
Varianza: Row(kurtosis(Amount_USD)=91251.1946216962)


- Hay mucha oferta de casa baratas con algunas casas muy caras
- Si  el valor de Kurtosis es mayor que tres es muy probable que haya datos extremos

# 3. Interpretacion para IA
## 3.1 Pre-procesamiento para redes neuronales

In [27]:
df = df.withColumn("Amount_norm", (f.col("Amount_USD") - media_amount) / desviacion_amount)
df = df.withColumn("Area_norm", (f.col("Area_m2") - media_area) / desviacion_area)
df.show(5)

+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+--------------------+--------------------+
|index|               Title|         Description|Amount(in rupees)|Price (in rupees)e|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|         Amount_norm|           Area_norm|
+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------

## 3.2 Gestión de outliers (Kurtosis)

In [42]:
q_h = df.approxQuantile("Amount_USD", [0.99], 0.01)[0]

df_limpio = df.filter(f.col("Amount_USD") <= q_h)

In [43]:
kurtosis_amount_limpio = df_limpio.select(f.kurtosis("Amount_USD")).collect()[0][0]
varianza_amount_limpio = df_limpio.select(f.variance("Amount_USD")).collect()[0][0]
media_amount_limpio = df_limpio.select(f.mean("Amount_USD")).collect()[0][0]

print(kurtosis_amount_limpio)
print(varianza_amount_limpio)
print(media_amount_limpio)

91251.1946216962
223974301803.79224
143776.13026927641


- La Kurtosis sigue siendo exatamente la misma que antes, que el el porcentaje que hemoes eliminado de datos es demasiado pequeño como para poder notar el efecto

# 4. Análisis de segmentos
## 4.1 Ingeniería de varible

In [44]:
df_limpio = df_limpio.withColumn(
    "Num_Bedrooms",
    f.split(f.col("Title")," ").getItem(0)
)
 
df_limpio = df_limpio.filter(f.col("Num_Bedrooms").isNotNull())
 
df_limpio = df_limpio.withColumn(
    "Num_Bedrooms",
    f.regexp_extract(f.col("Num_Bedrooms"), r"(\d+)", 1).cast("int")
)
 
df_limpio.show(5)

+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+--------------------+----------+----------+---------+----------+------------------+--------------------+--------------------+------------+
|index|               Title|         Description|Amount(in rupees)|Price (in rupees)e|location|Carpet Area|       Status|       Floor|Transaction|    Furnishing|facing|         overlooking|             Society|Bathroom|Balcony|Car Parking|           Ownership|Super Area|Dimensions|Plot Area|Amount_USD|           Area_m2|         Amount_norm|           Area_norm|Num_Bedrooms|
+-----+--------------------+--------------------+-----------------+------------------+--------+-----------+-------------+------------+-----------+--------------+------+--------------------+--------------------+--------+-------+-----------+-----

## 4.2 Cálculo de estadísticas por grupo

In [47]:
bed = df_limpio.groupBy("Num_Bedrooms").agg(
    f.mean("Amount_USD"),
    f.variance("Amount_USD"),
    f.skewness("Amount_USD")
)
bed.show(truncate=False)

df_limpio = df_limpio.join(bed, "Num_Bedrooms")

+------------+------------------+---------------------+--------------------+
|Num_Bedrooms|avg(Amount_USD)   |var_samp(Amount_USD) |skewness(Amount_USD)|
+------------+------------------+---------------------+--------------------+
|1           |42374.56647398844 |1.0420865682176313E9 |4.0605155154248225  |
|6           |844643.4782608695 |8.294408613855709E11 |1.8175914038274064  |
|3           |165414.9269135928 |4.1883196135625726E11|232.51924466950766  |
|5           |556257.212971078  |2.6273427355748886E11|6.789259923400493   |
|9           |436285.71428571426|3.6380571428571434E10|-0.2051309353148022 |
|4           |367763.79285014694|8.664553059243298E10 |4.191841836575597   |
|8           |2507820.0         |3.7904216256E12      |0.06164867132208699 |
|7           |682742.8571428572 |5.269206925714285E11 |1.8506352685940017  |
|10          |796254.5454545454 |1.8588609687272727E12|2.6012315773520585  |
|2           |74568.32623871956 |4.521417510532481E10 |189.30639791419486  |

- A: El valor de la desviacion se dispara cuanto mas grande es la vivienda. En 3bhk son vivienda y a consideradas de lujo mientras que las de 1bhk son viviendas normales

- B: En 3 BHK, la desviación es tan alta que el promedio no representa a casi nadie, el riesgo de tasación errónea es altísimo por la dispersión de precios.

- C: Segmento con mayor desviación/asimetría: El de 3 BHK

- C: ¿Representan la realidad?: No, Son excepciones que distorsionan la visión general.Deberían analizarse en un estudio de mercado aparte (segmento Luxury) para que el promedio del barrio sea útil para el ciudadano común.